- **Name:** 20.5_streaming_aggregations
- **Author:** Shamas Imran
- **Desciption:** Performing aggregations in structured streaming
- **Date:** 19-Aug-2025
<!--
REVISION HISTORY
Version          Date        Author           Desciption
01           19-Aug-2025   Shamas Imran       Applied groupBy aggregations on streams  
                                              Used tumbling and sliding windows  
                                              Demonstrated append vs update modes  
-->

In [11]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampType
from pyspark.sql.functions import col, window, session_window

StatementMeta(, 8acb8b0f-15b7-4ee2-8cc6-cf75e9755d57, 13, Finished, Available, Finished)

In [12]:
# ------------------------------------------------------------
# 1) Spark Session
# ------------------------------------------------------------
spark = (
    SparkSession.builder
        .appName("Aggregations_and_WindowedAggregations")
        .getOrCreate()
)

StatementMeta(, 8acb8b0f-15b7-4ee2-8cc6-cf75e9755d57, 14, Finished, Available, Finished)

In [13]:
rootPath = "abfss://shamas_ws@onelake.dfs.fabric.microsoft.com/test_Lakehouse.Lakehouse/Files/"
masterPath = rootPath + "spark-streaming/"
inputPath       = masterPath + "csv_input"
checkpointPath  = masterPath + "checkpoints/aggregations"
outputPath      = masterPath + "csv_output"

StatementMeta(, 8acb8b0f-15b7-4ee2-8cc6-cf75e9755d57, 15, Finished, Available, Finished)

In [14]:
# Check if folder exists before deleting
if mssparkutils.fs.exists(masterPath):
    mssparkutils.fs.rm(masterPath, True)  # True = recursive delete

# Create directories
import os

os.makedirs(masterPath, exist_ok=True)
os.makedirs(inputPath, exist_ok=True)
os.makedirs(checkpointPath, exist_ok=True)
os.makedirs(outputPath, exist_ok=True)

StatementMeta(, 8acb8b0f-15b7-4ee2-8cc6-cf75e9755d57, 16, Finished, Available, Finished)

In [15]:
import pandas as pd
import os

lakehouse_folder = inputPath  # your Lakehouse input path

data = [
    {"id": 1,  "name": "Ahsan", "score": 85, "event_time": "2025-08-18 10:00:00"},
    {"id": 2,  "name": "Sana",  "score": 92, "event_time": "2025-08-18 10:01:00"},
    {"id": 3,  "name": "Ali",   "score": 78, "event_time": "2025-08-18 10:02:00"},
    {"id": 4,  "name": "Ahsan", "score": 88, "event_time": "2025-08-18 10:03:00"},
    {"id": 5,  "name": "Sana",  "score": 95, "event_time": "2025-08-18 10:04:00"},
    {"id": 6,  "name": "Ali",   "score": 82, "event_time": "2025-08-18 10:06:00"},
    {"id": 7,  "name": "Ahsan", "score": 90, "event_time": "2025-08-18 10:07:00"},
    {"id": 8,  "name": "Sana",  "score": 87, "event_time": "2025-08-18 10:09:00"},
    {"id": 9,  "name": "Ali",   "score": 91, "event_time": "2025-08-18 10:10:00"},
    {"id": 10, "name": "Ahsan", "score": 80, "event_time": "2025-08-18 09:50:00"},  # Late, within 10-min watermark
    {"id": 11, "name": "Sana",  "score": 85, "event_time": "2025-08-18 09:40:00"},  # Too late, will be dropped by watermark
    {"id": 12, "name": "Ali",   "score": 88, "event_time": "2025-08-18 10:12:00"},
    {"id": 13, "name": "Ahsan", "score": 92, "event_time": "2025-08-18 10:14:00"},
    {"id": 14, "name": "Sana",  "score": 90, "event_time": "2025-08-18 10:15:00"},
    {"id": 15, "name": "Ali",   "score": 83, "event_time": "2025-08-18 10:18:00"},
]

valid_df = pd.DataFrame(data)
valid_path = os.path.join(lakehouse_folder, "Valid_data.csv")
valid_df.to_csv(valid_path, index=False, header=True)

StatementMeta(, 8acb8b0f-15b7-4ee2-8cc6-cf75e9755d57, 17, Finished, Available, Finished)

In [16]:
df = spark.read.format("csv").option("header","true").load(f"{inputPath}/Valid_data.csv")
df.show()

StatementMeta(, 8acb8b0f-15b7-4ee2-8cc6-cf75e9755d57, 18, Finished, Available, Finished)

+---+-----+-----+-------------------+
| id| name|score|         event_time|
+---+-----+-----+-------------------+
|  1|Ahsan|   85|2025-08-18 10:00:00|
|  2| Sana|   92|2025-08-18 10:01:00|
|  3|  Ali|   78|2025-08-18 10:02:00|
|  4|Ahsan|   88|2025-08-18 10:03:00|
|  5| Sana|   95|2025-08-18 10:04:00|
|  6|  Ali|   82|2025-08-18 10:06:00|
|  7|Ahsan|   90|2025-08-18 10:07:00|
|  8| Sana|   87|2025-08-18 10:09:00|
|  9|  Ali|   91|2025-08-18 10:10:00|
| 10|Ahsan|   80|2025-08-18 09:50:00|
| 11| Sana|   85|2025-08-18 09:40:00|
| 12|  Ali|   88|2025-08-18 10:12:00|
| 13|Ahsan|   92|2025-08-18 10:14:00|
| 14| Sana|   90|2025-08-18 10:15:00|
| 15|  Ali|   83|2025-08-18 10:18:00|
+---+-----+-----+-------------------+



In [17]:
# ------------------------------------------------------------
# 3) Define Schema
# ------------------------------------------------------------
schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("score", IntegerType(), True),
    StructField("event_time", TimestampType(), True)
])

# ------------------------------------------------------------
# 4) Create Streaming DataFrame
# ------------------------------------------------------------
df_stream = (
    spark.readStream
         .option("header", "true")
         .schema(schema)
         .csv(inputPath)
)

StatementMeta(, 8acb8b0f-15b7-4ee2-8cc6-cf75e9755d57, 19, Finished, Available, Finished)

In [18]:
# ------------------------------------------------------------
# 5) Aggregation WITHOUT Watermark
# ------------------------------------------------------------
# Stateful aggregation without watermark (state can grow indefinitely)
agg_no_watermark = df_stream.groupBy("name").count()

agg_no_watermark_query = (
    agg_no_watermark.writeStream
                .format("delta")                       # ✅ Delta table sink
                .option("checkpointLocation", checkpointPath + "/no_watermark")
                .outputMode("complete")                # required for aggregations
                .trigger(processingTime="10 seconds")  # micro-batch interval
                .toTable("agg_no_watermark_query")
)


# ------------------------------------------------------------
# 6) Aggregation WITH Watermark
# ------------------------------------------------------------
# Keep state for 10 minutes; late events beyond 10 min are dropped
from pyspark.sql.functions import window

# Add watermark
df_watermarked = df_stream.withWatermark("event_time", "10 minutes")

# Group by window + name
agg_with_watermark = (
    df_watermarked
        .groupBy(window("event_time", "10 minutes"), "name")
        .count()
        .orderBy("window", "name")
)

agg_with_watermark_query = (
    agg_with_watermark.writeStream
                .format("delta")                       # ✅ Delta table sink
                .option("checkpointLocation", checkpointPath + "/with_watermark")
                .outputMode("complete")                # required for aggregations
                .trigger(processingTime="10 seconds")  # micro-batch interval
                .toTable("agg_with_watermark_query")
)

# ------------------------------------------------------------
# 8) Wait for Completion
# ------------------------------------------------------------
spark.streams.awaitAnyTermination()

StatementMeta(, 8acb8b0f-15b7-4ee2-8cc6-cf75e9755d57, 20, Finished, Available, Finished)

StreamingQueryException: [STREAM_FAILED] Query [id = bf9bf670-4c94-4cda-bb06-224d389c5418, runId = 289d39ad-e353-4282-8dc8-46eb809eafc6] terminated with exception: [_LEGACY_ERROR_TEMP_DELTA_0007] A schema mismatch detected when writing to the Delta table (Table ID: 18ac8361-22f9-4f50-95ac-a16613cde2d0).
To enable schema migration using DataFrameWriter or DataStreamWriter, please set:
'.option("mergeSchema", "true")'.
For other operations, set the session configuration
spark.databricks.delta.schema.autoMerge.enabled to "true". See the documentation
specific to the operation for details.

Table schema:
root
-- name: string (nullable = true)
-- count: long (nullable = true)


Data schema:
root
-- window: struct (nullable = true)
    |-- start: timestamp (nullable = true)
    |-- end: timestamp (nullable = true)
-- name: string (nullable = true)
-- count: long (nullable = true)

         
To overwrite your schema or change partitioning, please set:
'.option("overwriteSchema", "true")'.

No

In [19]:
df = spark.sql("SELECT * FROM test_Lakehouse.agg_no_watermark_query LIMIT 1000")
display(df)

df = spark.sql("SELECT * FROM test_Lakehouse.agg_with_watermark_query LIMIT 1000")
display(df)

StatementMeta(, 8acb8b0f-15b7-4ee2-8cc6-cf75e9755d57, 21, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 3a0c2674-d44d-4f41-9bae-285ade833789)

SynapseWidget(Synapse.DataFrame, 92af5b72-599a-41d7-859a-728e2b2a7200)

In [0]:
# ------------------------------------------------------------
# 7) Tumbling Window Aggregation
# ------------------------------------------------------------
# Fixed-size, non-overlapping 5-minute windows
tumbling_window_agg = (
    df_watermarked.groupBy(window(col("event_time"), "5 minutes"), col("name"))
                   .count()
                   .orderBy("window")
)

tumbling_window_query = (
    tumbling_window_agg.writeStream
                       .format("console")
                       .option("checkpointLocation", checkpointPath + "/tumbling_window")
                       .outputMode("update")
                       .start()
)

In [0]:
# ------------------------------------------------------------
# 8) Sliding Window Aggregation
# ------------------------------------------------------------
# Fixed-size 5-min windows, sliding every 1 minute
sliding_window_agg = (
    df_watermarked.groupBy(window(col("event_time"), "5 minutes", "1 minute"), col("name"))
                   .count()
                   .orderBy("window")
)

sliding_window_query = (
    sliding_window_agg.writeStream
                      .format("console")
                      .option("checkpointLocation", checkpointPath + "/sliding_window")
                      .outputMode("update")
                      .start()
)


In [0]:
# ------------------------------------------------------------
# 9) Session Window Aggregation
# ------------------------------------------------------------
# Dynamic window based on inactivity gap of 5 minutes
session_window_agg = (
    df_watermarked.groupBy(session_window(col("event_time"), "5 minutes"), col("name"))
                   .count()
                   .orderBy("session_window")
)

session_window_query = (
    session_window_agg.writeStream
                      .format("console")
                      .option("checkpointLocation", checkpointPath + "/session_window")
                      .outputMode("update")
                      .start()
)

# ------------------------------------------------------------
# 10) Wait for All Streams
# ------------------------------------------------------------
spark.streams.awaitAnyTermination()

In [0]:
id,name,score,event_time
1,Ahsan,85,2025-08-18 10:00:00
2,Sana,92,2025-08-18 10:01:00
3,Ali,78,2025-08-18 10:02:00
4,Ahsan,88,2025-08-18 10:03:00
5,Sana,95,2025-08-18 10:04:00
6,Ali,82,2025-08-18 10:06:00
7,Ahsan,90,2025-08-18 10:07:00
8,Sana,87,2025-08-18 10:09:00
9,Ali,91,2025-08-18 10:10:00
10,Ahsan,80,2025-08-18 09:50:00   # Late, within 10-min watermark
11,Sana,85,2025-08-18 09:40:00    # Too late, will be dropped by watermark
12,Ali,88,2025-08-18 10:12:00
13,Ahsan,92,2025-08-18 10:14:00
14,Sana,90,2025-08-18 10:15:00
15,Ali,83,2025-08-18 10:18:00